# 03 · Álgebra lineal para Machine Learning, desde cero

**Módulo 1 · Sesión 3** — Fundamentos matemáticos

## Objetivos

No es un curso de álgebra lineal: es el subconjunto que de verdad se usa en Machine
Learning, y siempre con la conexión explícita al algoritmo donde reaparece.

1. Representar datos como vectores y matrices.
2. Producto punto: la operación que está debajo de casi todo modelo lineal.
3. Normas y distancias: cómo se mide "parecido" (base de KNN y de la regularización).
4. La matriz de covarianza y sus vectores propios.
5. SVD: la descomposición sobre la que se construye PCA.

| Concepto | Dónde reaparece |
|---|---|
| Producto punto | Regresión lineal y logística (S6, S9), redes neuronales (S13) |
| Normas $L_1$ y $L_2$ | Regularización Lasso y Ridge (S7) |
| Distancias | KNN (S9), K-Means y DBSCAN (S12) |
| Vectores propios | PCA (S12) |
| SVD | PCA (S12), compresión y reducción de ruido |

## Paquetes

`numpy`, `pandas`, `matplotlib`.

In [ ]:
# Arranque para Google Colab (en local no hace nada): trae el repositorio para que
# ../datos y ../src existan. Ejecútala antes que cualquier otra celda.
import sys
if "google.colab" in sys.modules:
    !git clone -q --depth 1 https://github.com/delany-ramirez/machine_learning /content/machine_learning
    %cd /content/machine_learning/modulo-1-fundamentos-ciclo-vida/notebooks

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEMILLA = 42
rng = np.random.default_rng(SEMILLA)

## 1. Un dato es un vector

La idea que conecta el álgebra lineal con el Machine Learning es simple: **cada observación
es un punto en un espacio de tantas dimensiones como variables tenga**.

Un estudiante con promedio 4.2, que estudia 12 horas semanales y asiste al 90 % de las
clases es el punto $(4.2,\ 12,\ 90)$ en un espacio de tres dimensiones.

In [ ]:
estudiante = np.array([4.2, 12.0, 90.0])

print("Vector:    ", estudiante)
print("Dimensión: ", estudiante.shape)
print("Tipo:      ", estudiante.dtype)

Y un conjunto de datos es una **matriz**: una fila por observación, una columna por
variable. Es la famosa matriz de diseño $\mathbf{X} \in \mathbb{R}^{n \times p}$, con $n$
observaciones y $p$ variables.

In [ ]:
X = np.array(
    [
        [4.2, 12.0, 90.0],
        [3.1, 4.0, 62.0],
        [3.6, 9.0, 85.0],
        [4.8, 20.0, 97.0],
    ]
)

print(f"X tiene {X.shape[0]} observaciones y {X.shape[1]} variables")
print(X)

Operaciones básicas de indexado: una fila es una observación, una columna es una variable.

In [ ]:
print("Primer estudiante (fila 0):  ", X[0])
print("Todas las asistencias (col 2):", X[:, 2])
print("Media por variable:           ", X.mean(axis=0).round(2))
print("Desviación por variable:      ", X.std(axis=0).round(2))

> **`axis=0` recorre las filas**, así que devuelve un resultado por columna (por variable).
> `axis=1` haría lo contrario. Es la fuente número uno de errores silenciosos en NumPy: no
> falla, simplemente calcula otra cosa.

## 2. El producto punto

Si hay una sola operación que entender, es esta. El producto punto de dos vectores
multiplica elemento a elemento y suma:

$$
\mathbf{a} \cdot \mathbf{b} = \sum_{i=1}^{p} a_i b_i
$$

In [ ]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])

manual = sum(a[i] * b[i] for i in range(len(a)))
print("A mano:  ", manual)
print("NumPy:   ", np.dot(a, b))
print("Operador:", a @ b)

### Por qué importa

**Una predicción lineal es un producto punto.** Cuando en el notebook 01 el modelo predijo
la nota de un estudiante, hizo exactamente esto:

$$
\hat{y} = \beta_0 + \boldsymbol{\beta} \cdot \mathbf{x}
$$

In [ ]:
beta_0 = -0.15
beta = np.array([0.62, 0.055, 0.011])  # promedio, horas, asistencia

prediccion = beta_0 + beta @ estudiante
print(f"Nota predicha para el estudiante {estudiante}: {prediccion:.2f}")

Y para **todos** los estudiantes a la vez, sin ningún bucle: multiplicar la matriz por el
vector de coeficientes.

$$
\hat{\mathbf{y}} = \beta_0 + \mathbf{X}\boldsymbol{\beta}
$$

In [ ]:
predicciones = beta_0 + X @ beta
print("Predicciones para los 4 estudiantes:", predicciones.round(2))

Esa es la razón de fondo por la que el Machine Learning se escribe en álgebra lineal:
**una sola operación matricial reemplaza un bucle sobre millones de filas**, y las
bibliotecas la ejecutan en código optimizado (y en GPU, cuando hace falta).

Midamos la diferencia.

In [ ]:
X_grande = rng.normal(size=(200_000, 3))

inicio = pd.Timestamp.now()
resultado_bucle = np.array([beta_0 + beta @ fila for fila in X_grande])
tiempo_bucle = (pd.Timestamp.now() - inicio).total_seconds()

inicio = pd.Timestamp.now()
resultado_matriz = beta_0 + X_grande @ beta
tiempo_matriz = (pd.Timestamp.now() - inicio).total_seconds()

print(f"Bucle de Python:      {tiempo_bucle:.4f} s")
print(f"Producto matricial:   {tiempo_matriz:.4f} s")
print(f"Aceleración:          {tiempo_bucle / max(tiempo_matriz, 1e-9):.0f}x")
print(f"¿Mismo resultado?     {np.allclose(resultado_bucle, resultado_matriz)}")

### Interpretación geométrica

El producto punto también mide **alineación** entre dos vectores:

$$
\mathbf{a} \cdot \mathbf{b} = \|\mathbf{a}\|\,\|\mathbf{b}\|\cos\theta
$$

Despejando el coseno obtenemos la **similitud coseno**, muy usada para comparar documentos
o *embeddings* de texto.

In [ ]:
def similitud_coseno(u, v):
    return (u @ v) / (np.linalg.norm(u) * np.linalg.norm(v))


paralelos = (np.array([1.0, 1.0]), np.array([2.0, 2.0]))
perpendiculares = (np.array([1.0, 0.0]), np.array([0.0, 1.0]))
opuestos = (np.array([1.0, 1.0]), np.array([-1.0, -1.0]))

for nombre, (u, v) in [
    ("Paralelos", paralelos),
    ("Perpendiculares", perpendiculares),
    ("Opuestos", opuestos),
]:
    cos = similitud_coseno(u, v)
    print(f"{nombre:16s} cos = {cos:6.2f}   ángulo = {np.degrees(np.arccos(np.clip(cos, -1, 1))):5.1f}°")

## 3. Normas: medir el tamaño de un vector

La **norma** es la longitud de un vector. Hay varias, y la elección tiene consecuencias
prácticas enormes.

$$
\|\mathbf{v}\|_1 = \sum_i |v_i|
\qquad\qquad
\|\mathbf{v}\|_2 = \sqrt{\sum_i v_i^2}
$$

In [ ]:
v = np.array([3.0, -4.0])

print(f"Norma L1 (Manhattan):  {np.linalg.norm(v, ord=1):.2f}")
print(f"Norma L2 (Euclidiana): {np.linalg.norm(v, ord=2):.2f}")
print(f"Verificación L2:       {np.sqrt(3**2 + 4**2):.2f}")

### Por qué importa: regularización

En la sesión 7 añadiremos una penalización sobre los coeficientes del modelo para evitar
que crezcan sin control:

- **Ridge** penaliza $\|\boldsymbol{\beta}\|_2^2$ → encoge los coeficientes hacia cero.
- **Lasso** penaliza $\|\boldsymbol{\beta}\|_1$ → lleva algunos coeficientes **exactamente**
  a cero, haciendo selección de variables.

Esa diferencia de comportamiento sale enteramente de la geometría de las dos normas. Aquí
ya se intuye: para un mismo "presupuesto" de norma, la $L_1$ prefiere las esquinas de los
ejes (donde alguna coordenada es cero) y la $L_2$ no tiene esquinas.

In [ ]:
angulos = np.linspace(0, 2 * np.pi, 400)
fig, eje = plt.subplots(figsize=(5, 5))

# Bola L2: circunferencia de radio 1.
eje.plot(np.cos(angulos), np.sin(angulos), label="$\\|v\\|_2 = 1$ (Ridge)")

# Bola L1: rombo de vértices en los ejes.
rombo = np.array([[1, 0], [0, 1], [-1, 0], [0, -1], [1, 0]], dtype=float)
eje.plot(rombo[:, 0], rombo[:, 1], label="$\\|v\\|_1 = 1$ (Lasso)")

eje.axhline(0, color="gray", linewidth=0.5)
eje.axvline(0, color="gray", linewidth=0.5)
eje.set_aspect("equal")
eje.set_title("Las esquinas de $L_1$ son las que anulan coeficientes")
eje.legend()
plt.tight_layout()
plt.show()

## 4. Distancias: la base de KNN y del clustering

La distancia entre dos observaciones es la norma de su diferencia:

$$
d(\mathbf{u}, \mathbf{v}) = \|\mathbf{u} - \mathbf{v}\|_2
$$

Buscar "los estudiantes más parecidos a este" es literalmente ordenar por esta distancia:
eso es K-Nearest Neighbors (S9), y también el criterio de asignación de K-Means (S12).

In [ ]:
consulta = np.array([4.0, 10.0, 88.0])
distancias = np.linalg.norm(X - consulta, axis=1)

tabla = pd.DataFrame(X, columns=["promedio", "horas", "asistencia"])
tabla["distancia"] = distancias.round(2)
print(f"Consulta: {consulta}\n")
print(tabla.sort_values("distancia").to_string(index=False))

### La trampa de las escalas

Mira los números: la asistencia va de 0 a 100, el promedio de 0 a 5. La distancia está
**dominada por la asistencia**, simplemente porque sus números son más grandes. El promedio
académico casi no influye, aunque sea la variable más informativa.

Comprobémoslo separando la contribución de cada variable.

In [ ]:
contribucion = ((X - consulta) ** 2)
aporte = pd.DataFrame(contribucion, columns=["promedio", "horas", "asistencia"]).round(2)
aporte["% asistencia"] = (
    100 * contribucion[:, 2] / contribucion.sum(axis=1)
).round(1)
print("Aporte de cada variable a la distancia al cuadrado:\n")
print(aporte.to_string(index=False))

La asistencia aporta la mayor parte de la distancia en casi todas las filas. **Por eso hay
que estandarizar antes de usar cualquier algoritmo basado en distancias.** Es una de las
reglas prácticas más importantes del curso, y la formalizaremos en la sesión 5.

Estandarizar es restar la media y dividir por la desviación estándar:

$$
z = \frac{x - \mu}{\sigma}
$$

In [ ]:
X_estandarizado = (X - X.mean(axis=0)) / X.std(axis=0)
consulta_estandarizada = (consulta - X.mean(axis=0)) / X.std(axis=0)

contribucion_z = (X_estandarizado - consulta_estandarizada) ** 2
aporte_z = pd.DataFrame(contribucion_z, columns=["promedio", "horas", "asistencia"]).round(2)
aporte_z["% asistencia"] = (
    100 * contribucion_z[:, 2] / contribucion_z.sum(axis=1)
).round(1)
print("Tras estandarizar, el aporte se reparte:\n")
print(aporte_z.to_string(index=False))

## 5. La matriz de covarianza

La covarianza mide cómo varían dos variables **juntas**. La matriz de covarianza recoge
todas las parejas: en la diagonal están las varianzas, fuera de ella las covarianzas.

$$
\mathbf{\Sigma} = \frac{1}{n-1}\,\mathbf{X}_c^{\top}\mathbf{X}_c
$$

donde $\mathbf{X}_c$ es la matriz con las columnas centradas (media cero). Usamos ahora el
dataset real del módulo.

In [ ]:
datos = pd.read_csv("../datos/rendimiento-estudiantes.csv")
variables = ["promedio_anterior", "horas_estudio_semana", "asistencia_pct", "nota_final"]
D = datos[variables].to_numpy()

D_centrada = D - D.mean(axis=0)
covarianza = (D_centrada.T @ D_centrada) / (len(D) - 1)

print("Matriz de covarianza calculada a mano:\n")
print(pd.DataFrame(covarianza, index=variables, columns=variables).round(3).to_string())
print(f"\n¿Coincide con np.cov? {np.allclose(covarianza, np.cov(D, rowvar=False))}")

Los números de la covarianza dependen de las unidades, así que cuesta interpretarlos. La
**correlación** es la covarianza estandarizada, siempre entre $-1$ y $1$.

In [ ]:
correlacion = np.corrcoef(D, rowvar=False)
print(pd.DataFrame(correlacion, index=variables, columns=variables).round(3).to_string())

## 6. Vectores y valores propios

Un **vector propio** de una matriz es un vector que la matriz no rota: solo lo estira o lo
encoge. Cuánto lo estira es su **valor propio**.

$$
\mathbf{\Sigma}\mathbf{v} = \lambda\mathbf{v}
$$

Aplicado a la matriz de covarianza, esto tiene un significado precioso: **los vectores
propios son las direcciones de máxima variabilidad de los datos, y los valores propios
dicen cuánta variabilidad hay en cada dirección**. Eso es exactamente PCA (sesión 12).

In [ ]:
valores, vectores = np.linalg.eigh(covarianza)

# eigh los devuelve en orden ascendente; los queremos de mayor a menor.
orden = np.argsort(valores)[::-1]
valores, vectores = valores[orden], vectores[:, orden]

varianza_explicada = valores / valores.sum()
resumen = pd.DataFrame(
    {
        "componente": [f"PC{i+1}" for i in range(len(valores))],
        "valor_propio": valores.round(3),
        "varianza_explicada": (varianza_explicada * 100).round(1),
        "acumulada": (np.cumsum(varianza_explicada) * 100).round(1),
    }
)
print(resumen.to_string(index=False))

La primera componente concentra la mayor parte de la variabilidad. Pero mira con cuidado la
matriz de covarianza de arriba: `asistencia_pct` tiene una varianza de ~78 y
`promedio_anterior` de ~0.23. **La primera componente no está capturando la estructura de
los datos, está capturando la variable con las unidades más grandes.**

Es el mismo problema de escalas de la sección 4, ahora mordiendo a PCA. Por eso PCA se
aplica casi siempre sobre datos estandarizados; lo haremos correctamente en la sesión 12.
Aquí lo dejamos crudo justamente para que veas el síntoma.

Verifiquemos ahora la definición de vector propio.

In [ ]:
v1 = vectores[:, 0]
print("Σv  =", (covarianza @ v1).round(4))
print("λv  =", (valores[0] * v1).round(4))
print("¿Son iguales?", np.allclose(covarianza @ v1, valores[0] * v1))

## 7. SVD: la descomposición general

La descomposición en valores singulares factoriza **cualquier** matriz (no hace falta que
sea cuadrada ni simétrica):

$$
\mathbf{X} = \mathbf{U}\mathbf{S}\mathbf{V}^{\top}
$$

- $\mathbf{V}$ contiene las direcciones principales (las mismas que los vectores propios de
  la covarianza).
- $\mathbf{S}$ contiene los valores singulares, que dicen cuánta información aporta cada
  dirección.

Es lo que scikit-learn usa por dentro para calcular PCA, porque es numéricamente más
estable que calcular la covarianza y diagonalizarla.

In [ ]:
U, S, Vt = np.linalg.svd(D_centrada, full_matrices=False)

print("Formas:  U", U.shape, " S", S.shape, " Vt", Vt.shape)
print("\nValores singulares:", S.round(2))

# La relación entre valores singulares y valores propios de la covarianza:
print("\nS^2/(n-1):    ", (S**2 / (len(D) - 1)).round(3))
print("Valores propios:", valores.round(3))

Coinciden: **SVD y la descomposición espectral de la covarianza son la misma cosa vista
desde dos ángulos**. Es la afirmación que sostiene toda la sesión 12.

### Aproximación de rango bajo

Si conservamos solo las primeras $k$ direcciones, obtenemos la mejor aproximación posible de
la matriz con esa cantidad de información. Eso es *comprimir*: menos números, casi los
mismos datos.

In [ ]:
print(f"{'k':>3} {'valores usados':>15} {'error relativo':>16}")
for k in range(1, 5):
    D_aprox = U[:, :k] @ np.diag(S[:k]) @ Vt[:k]
    error = np.linalg.norm(D_centrada - D_aprox) / np.linalg.norm(D_centrada)
    print(f"{k:>3} {k * (len(D) + len(variables)):>15,} {error:>15.2%}")

print(f"\nMatriz original: {D_centrada.size:,} valores")

Con dos direcciones de cuatro reconstruimos los datos con menos del 7 % de error, usando la
mitad de los números. Esa es la idea de la reducción de dimensionalidad, y la razón por la
que PCA funciona: **la información real de un conjunto de datos suele vivir en muchas menos
dimensiones de las que tiene la tabla**.

## Resumen

| Herramienta | Qué hace | Dónde vuelve a aparecer |
|---|---|---|
| Producto punto | Combina variables con pesos | Toda predicción lineal (S6, S7, S9, S13) |
| Producto matricial | Predice sobre todo el dataset sin bucles | Todo el curso |
| Norma $L_2$ | Longitud euclidiana | Ridge (S7), distancias |
| Norma $L_1$ | Suma de valores absolutos | Lasso (S7) |
| Distancia | Cuán parecidas son dos observaciones | KNN (S9), K-Means, DBSCAN (S12) |
| Covarianza | Cómo varían juntas dos variables | Multicolinealidad (S7), PCA (S12) |
| Vectores propios | Direcciones de máxima variabilidad | PCA (S12) |
| SVD | Factorización general y compresión | PCA (S12) |

**La regla práctica que más te va a servir:** todo lo que use distancias necesita variables
estandarizadas.

## Para practicar

1. Calcula a mano la similitud coseno entre el estudiante `E0001` y `E0002` del dataset,
   usando las cuatro variables numéricas. ¿Cambia el resultado si estandarizas antes?
2. Añade `edad` y `estrato` al cálculo de la matriz de covarianza. ¿Cuánta varianza explica
   ahora la primera componente? ¿Por qué cambia?
3. Verifica que los vectores propios son ortogonales entre sí: su producto punto debe ser
   cero. ¿Por qué la matriz de covarianza garantiza esa propiedad?
4. Reconstruye los datos con $k=2$ y compara la columna `nota_final` reconstruida con la
   original. ¿Qué estudiantes quedan peor representados?